<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/transpose_shared.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:


from numba.core.types import float32
from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32

@cuda.jit
def transpose (input_mat, output_mat):
  n,m = output_mat.shape

  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

  if row < n and col < m:
    output_mat[row,col] = input_mat[col,row]



def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1, 1, size=(m, n), dtype=cp.float32)
  output_mat = cp.zeros((n, m), dtype= cp.float32)

  #launch configuration
  # no. of elements in smem = no. of threads in block
  threads_per_block = (TILE_DIM, TILE_DIM)
  blocks_per_grid = (
      (m + (TILE_DIM - 1)) // TILE_DIM,
      (n + (TILE_DIM - 1)) // TILE_DIM
  )

  transpose[blocks_per_grid,threads_per_block] (input_mat,output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)

  np.testing.assert_allclose(output_mat_np, input_mat_np.T)

  print ("input matrix")
  print (input_mat_np)
  print ("output matrix")
  print (output_mat_np)

main()

input matrix
[[-0.3252104  -0.9944425   0.9476758  ... -0.8631738  -0.03447789
   0.08791113]
 [-0.04170179 -0.97812307 -0.1433574  ... -0.98800653 -0.12150097
   0.95153606]
 [ 0.36625993  0.31759298 -0.8611026  ...  0.6648021  -0.8267094
  -0.5868819 ]
 ...
 [-0.42035925  0.30564344  0.00453413 ... -0.79637676 -0.02732724
   0.06406379]
 [-0.05411863 -0.8090101  -0.7768712  ...  0.18749976 -0.44722134
  -0.5614867 ]
 [-0.9063536   0.71449757  0.00133383 ...  0.7843119  -0.5898299
  -0.57829076]]
output matrix
[[-0.3252104  -0.04170179  0.36625993 ... -0.42035925 -0.05411863
  -0.9063536 ]
 [-0.9944425  -0.97812307  0.31759298 ...  0.30564344 -0.8090101
   0.71449757]
 [ 0.9476758  -0.1433574  -0.8611026  ...  0.00453413 -0.7768712
   0.00133383]
 ...
 [-0.8631738  -0.98800653  0.6648021  ... -0.79637676  0.18749976
   0.7843119 ]
 [-0.03447789 -0.12150097 -0.8267094  ... -0.02732724 -0.44722134
  -0.5898299 ]
 [ 0.08791113  0.95153606 -0.5868819  ...  0.06406379 -0.5614867
  -0.57829

In [4]:
%%writefile transpose-naive.py



from numba.core.types import float32
from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32

@cuda.jit
def transpose (input_mat, output_mat):
  n,m = output_mat.shape

  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

  if row < n and col < m:
    output_mat[row,col] = input_mat[col,row]



def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1, 1, size=(m, n), dtype=cp.float32)
  output_mat = cp.zeros((n, m), dtype= cp.float32)

  #launch configuration
  # no. of elements in smem = no. of threads in block
  threads_per_block = (TILE_DIM, TILE_DIM)
  blocks_per_grid = (
      (m + (TILE_DIM - 1)) // TILE_DIM,
      (n + (TILE_DIM - 1)) // TILE_DIM
  )

  transpose[blocks_per_grid,threads_per_block] (input_mat,output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)

  np.testing.assert_allclose(output_mat_np, input_mat_np.T)

  print ("input matrix")
  print (input_mat_np)
  print ("output matrix")
  print (output_mat_np)

main()

Writing transpose-naive.py


In [5]:
!ncu --set full -f -o transpose-naive-rep python3 transpose-naive.py

==PROF== Connected to process 4710 (/usr/bin/python3.13)
==PROF== Profiling "generate_seed_pseudo" - 0: 0%....50%....100% - 43 passes
==PROF== Profiling "gen_sequenced" - 1: 0%....50%....100% - 43 passes
==PROF== Profiling "cupy_random_x_mod_1" - 2: 0%....50%....100% - 43 passes
==PROF== Profiling "cupy_scale" - 3: 0%....50%....100% - 43 passes
==PROF== Profiling "transpose[abi:v1,cw51cXTLSUwv..." - 4: 0%....50%....100% - 43 passes
input matrix
[[-0.4542485   0.839766    0.88245595 ...  0.2556529   0.45828605
  -0.41671848]
 [-0.9666998  -0.39785928 -0.5161978  ... -0.25915337  0.01722908
   0.0147481 ]
 [-0.263915   -0.63895094 -0.664487   ...  0.529078    0.8488797
  -0.16718048]
 ...
 [-0.24097562 -0.49458945  0.78088236 ...  0.69396317 -0.72421575
   0.17464614]
 [ 0.48661733  0.06738675  0.8386835  ... -0.6228307  -0.78988373
  -0.55781484]
 [-0.89824885 -0.7000314   0.08016133 ... -0.694729   -0.3372243
   0.20632672]]
output matrix
[[-0.4542485  -0.9666998  -0.263915   ... -0.24

In [6]:
print ("Shared..")

Shared..


In [7]:
%%writefile transpose-shared.py

from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32
TILE_DIM_ONE = TILE_DIM + 1

@cuda.jit
def transpose(input_mat, output_mat):
    n, m = output_mat.shape
    tile = cuda.shared.array((TILE_DIM, TILE_DIM_ONE), dtype=cp.float32)

    # Phase 1
    # copy from input_mat (gmem) to tile (smem)
    in_col = cuda.threadIdx.x + cuda.blockIdx.y * cuda.blockDim.x # type: ignore
    in_row = cuda.threadIdx.y + cuda.blockIdx.x * cuda.blockDim.y # type: ignore
    if in_row < m and in_col < n:
        tile[cuda.threadIdx.y, cuda.threadIdx.x] = input_mat[in_row, in_col]

    cuda.syncthreads()

    # Phase 2
    # copy from tile (smem) to output_mat (gmem)
    out_col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x # type: ignore
    out_row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y # type: ignore
    if out_row < n and out_col < m:
        output_mat[out_row, out_col] = tile[cuda.threadIdx.x, cuda.threadIdx.y]


def main():
    m = 10000
    n = 1200

    input_mat = cp.random.uniform(-1, 1, (m, n), dtype=cp.float32)
    output_mat = cp.zeros((n, m), dtype=cp.float32)

    # launch configuration
    threads_per_block = (TILE_DIM, TILE_DIM)
    blocks_per_grid = (
        (m + threads_per_block[0] - 1) // threads_per_block[0],
        (n + threads_per_block[1] - 1) // threads_per_block[1],
    )
    transpose[blocks_per_grid, threads_per_block](input_mat, output_mat)
    cuda.synchronize()

    input_mat_np = cp.asnumpy(input_mat)
    output_mat_np = cp.asnumpy(output_mat)
    np.testing.assert_allclose(input_mat_np.T, output_mat_np, rtol=1e-5, atol=1e-5)

main()

Overwriting transpose-shared.py


In [8]:
!ncu --set full -f -o transpose-shared-rep python3 transpose-shared.py

==PROF== Connected to process 5064 (/usr/bin/python3.13)
==PROF== Profiling "generate_seed_pseudo" - 0: 0%....50%....100% - 43 passes
==PROF== Profiling "gen_sequenced" - 1: 0%....50%....100% - 43 passes
==PROF== Profiling "cupy_random_x_mod_1" - 2: 0%....50%....100% - 43 passes
==PROF== Profiling "cupy_scale" - 3: 0%....50%....100% - 43 passes
==PROF== Profiling "transpose[abi:v1,cw51cXTLSUwv..." - 4: 0%....50%....100% - 43 passes
==PROF== Disconnected from process 5064
==PROF== Report: /content/transpose-shared-rep.ncu-rep
